# 🎬 CloneCinema — ComfyUI + WAN 2.2 TI2V 5B (Kaggle T4 x2)

**Modo Warm Session con Auto-Apagado Inteligente (Watchdog)**

- **Dataset Instantáneo**: Los pesos (~17 GB) se montan en 0 segundos desde `/kaggle/input/wan-2-2-5b-model-data`.
- **Autorregistro en Supabase**: Publica automáticamente la URL del túnel Cloudflare en la base de datos de CloneCinema.
- **Protección de 30 Horas (Watchdog)**: Si pasan 15 minutos sin generar video, el servidor se auto-apaga para no gastar cuota de GPU.
- **Bucket Supabase**: Soporta recepción de imágenes vía URL de Supabase (`wan_inputs` o `keyframes`).

In [ ]:
# ⚡ [1/5] Clonar ComfyUI e instalar dependencias
import os, sys, subprocess

print("⚡ [1/5] Instalando dependencias base y ComfyUI en /kaggle/working...")
subprocess.run(["apt-get", "update", "-qq"], stdout=subprocess.DEVNULL)
subprocess.run(["apt-get", "install", "-y", "aria2", "-qq"], stdout=subprocess.DEVNULL)

os.chdir("/kaggle/working")
if not os.path.exists("/kaggle/working/ComfyUI"):
    subprocess.run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", "/kaggle/working/ComfyUI"], check=True)

os.chdir("/kaggle/working/ComfyUI")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"], check=True)

# Instalar nodos esenciales
os.chdir("/kaggle/working/ComfyUI/custom_nodes")
nodes = [
    ("ComfyUI-Manager", "https://github.com/ltdrdata/ComfyUI-Manager.git"),
    ("ComfyUI-VideoHelperSuite", "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git")
]
for name, repo in nodes:
    if not os.path.exists(name):
        subprocess.run(["git", "clone", repo, name, "-q"], check=True)

os.chdir("/kaggle/working/ComfyUI")
print("✅ ComfyUI y dependencias listos en /kaggle/working/ComfyUI.")

In [ ]:
# 📥 [2/5] Carga ultrarrápida de modelos Wan 2.2 desde Dataset Kaggle
import os, subprocess, shutil

os.chdir("/kaggle/working/ComfyUI")
os.makedirs("models/diffusion_models", exist_ok=True)
os.makedirs("models/vae", exist_ok=True)
os.makedirs("models/text_encoders", exist_ok=True)
os.makedirs("input", exist_ok=True)

dataset_path = "/kaggle/input/wan-2-2-5b-model-data"

if os.path.exists(dataset_path):
    print("⚡ ¡Dataset Kaggle detectado! Enlazando pesos instantáneamente (0 segundos)...")
    # 1. Diffusion Model (wan2.2_ti2v_5B_fp16.safetensors)
    src_diff = os.path.join(dataset_path, "wan2.2_ti2v_5B_fp16.safetensors")
    dst_diff = "models/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors"
    if os.path.exists(src_diff) and not os.path.exists(dst_diff):
        os.symlink(src_diff, dst_diff)
        print("   🔗 Diffusion Model enlazado.")

    # 2. VAE (wan2.2_vae.safetensors)
    src_vae = os.path.join(dataset_path, "wan2.2_vae.safetensors")
    dst_vae = "models/vae/wan2.2_vae.safetensors"
    if os.path.exists(src_vae) and not os.path.exists(dst_vae):
        os.symlink(src_vae, dst_vae)
        print("   🔗 VAE enlazado.")

    # 3. Text Encoder (umt5_xxl_fp8_e4m3fn_scaled.safetensors)
    src_clip = os.path.join(dataset_path, "umt5_xxl_fp8_e4m3fn_scaled.safetensors")
    dst_clip = "models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors"
    if os.path.exists(src_clip) and not os.path.exists(dst_clip):
        os.symlink(src_clip, dst_clip)
        print("   🔗 Text Encoder T5 enlazado.")
    print("✅ Todos los modelos Wan 2.2 enlazados en 0.05 segundos.")
else:
    print("⚠️ Dataset no encontrado en /kaggle/input, descargando vía aria2c multihilo...")
    downloads = [
        ("models/diffusion_models", "wan2.2_ti2v_5B_fp16.safetensors", "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors"),
        ("models/vae", "wan2.2_vae.safetensors", "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors"),
        ("models/text_encoders", "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors")
    ]
    for out_dir, filename, url in downloads:
        dest = os.path.join(out_dir, filename)
        if not os.path.exists(dest):
            print(f"🚀 Descargando {filename}...")
            subprocess.run(["aria2c", "-c", "-x", "16", "-s", "16", "-k", "1M", "--file-allocation=none", "-d", out_dir, "-o", filename, url], check=True)


In [ ]:
# 🌐 [3/5] Instalar binario de Cloudflare Tunnel
import os, subprocess
os.chdir("/kaggle/working/ComfyUI")
if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "cloudflared"], check=True)
print("✅ Cloudflared listo en /kaggle/working/ComfyUI.")

In [ ]:
# 🚀 [4/5] Levantar servidor ComfyUI en GPU
import subprocess, time, os, sys
os.chdir("/kaggle/working/ComfyUI")
print("🚀 Levantando servidor ComfyUI en GPU T4...")
subprocess.Popen([sys.executable, "main.py", "--listen", "0.0.0.0", "--port", "8188", "--highvram", "--fast"])
time.sleep(12)
subprocess.run(["curl", "-s", "-o", "/dev/null", "-w", "ComfyUI HTTP Status: %{http_code}\n", "http://localhost:8188"])

In [ ]:
# 🛡️ [5/5] Túnel Cloudflare + Autorregistro en Supabase + Watchdog 2.0 (Inteligencia Superior)
import subprocess, time, re, json, urllib.request, threading, os, sys, base64

SUPABASE_URL = "https://micjwzatshemmovaneix.supabase.co"
try:
    from kaggle_secrets import UserSecretsClient
    SUPABASE_KEY = UserSecretsClient().get_secret("SUPABASE_SERVICE_ROLE_KEY")
except Exception:
    SUPABASE_KEY = os.environ.get("SUPABASE_KEY") or base64.b64decode("c2Jfc2VjcmV0XzFPOXV4QnoyM080cEhUYWxXVkdObHdfYU5HT09MQjU=").decode("utf-8")

TUNNEL_NODE_ID = "a0000000-0000-0000-0000-000000000001"
shutdown_event = threading.Event()

# --- Watchdog 2.0: Tiempos Dinámicos ---
INITIAL_TIMEOUT_MINUTES = 35       # Fase 1: Arranque y Setup inicial
OPERATIONAL_TIMEOUT_MINUTES = 15   # Fase 2: Régimen activo tras el primer render
WARNING_WINDOW_SECONDS = 120       # 2 minutos de preaviso interactivo al frontend

def register_supabase(tunnel_url, status="running", note="", extra=None):
    try:
        payload = {
            "id": TUNNEL_NODE_ID,
            "kind": "video",
            "status": status,
            "provider": "wan22_tunnel",
            "input_url": tunnel_url,
            "prompt": f"Kaggle Wan 2.2 Live Gateway - {note}",
            "updated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        }
        if extra and isinstance(extra, dict):
            payload.update(extra)

        req = urllib.request.Request(
            f"{SUPABASE_URL}/rest/v1/ai_jobs?on_conflict=id",
            data=json.dumps(payload).encode("utf-8"),
            headers={
                "apikey": SUPABASE_KEY,
                "Authorization": f"Bearer {SUPABASE_KEY}",
                "Content-Type": "application/json",
                "Prefer": "resolution=merge-duplicates"
            },
            method="POST"
        )
        with urllib.request.urlopen(req, timeout=10) as resp:
            pass
    except Exception as e:
        print(f"⚠️ Error sincronizando con Supabase: {e}")

def write_audit_session_log(tunnel_url, started_at, total_videos, reason):
    try:
        dur_min = round((time.time() - started_at) / 60.0, 1)
        audit_payload = {
            "id": f"sess_{int(started_at)}_{int(time.time())}",
            "kind": "gpu_session",
            "provider": "wan22_tunnel",
            "status": "done",
            "input_url": tunnel_url,
            "credits_charged": dur_min,
            "prompt": json.dumps({
                "session_type": "kaggle_comfyui_wan22",
                "started_at": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime(started_at)),
                "ended_at": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
                "duration_minutes": dur_min,
                "videos_rendered": total_videos,
                "shutdown_reason": reason
            }),
            "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        }
        req = urllib.request.Request(
            f"{SUPABASE_URL}/rest/v1/ai_jobs",
            data=json.dumps(audit_payload).encode("utf-8"),
            headers={
                "apikey": SUPABASE_KEY,
                "Authorization": f"Bearer {SUPABASE_KEY}",
                "Content-Type": "application/json"
            },
            method="POST"
        )
        urllib.request.urlopen(req, timeout=10)
        print(f"📜 [Audit SaaS] Registro de sesión guardado: {dur_min} min de GPU consumidos | {total_videos} videos | Motivo: {reason}")
    except Exception as e:
        print(f"⚠️ Error guardando registro de auditoría SaaS: {e}")

subprocess.run(["pkill", "-f", "cloudflared"])
proc = subprocess.Popen(
    ["/kaggle/working/ComfyUI/cloudflared", "tunnel", "--url", "http://localhost:8188"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

url = None
for _ in range(120):
    line = proc.stdout.readline()
    if not line:
        time.sleep(0.3)
        continue
    m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break

if not url:
    print("❌ No se pudo obtener la URL de Cloudflare.")
    sys.exit(1)

started_session_at = time.time()
print("\n" + "="*70)
print("🎉 ¡COMFYUI WAN 2.2 TI2V 5B ACTIVO!")
print(f"👉 ENLACE PÚBLICO: {url}")
print("="*70 + "\n")

# Registrar URL en Supabase con estado inicial
register_supabase(url, status="running", note="Activo y listo para generar (Fase Arranque: 35 min)")

# Watchdog 2.0 Loop
def watchdog_loop():
    last_job_count = 0
    total_videos_rendered = 0
    last_activity = time.time()
    has_rendered_any = False
    warning_emitted = False

    print(f"🛡️ [Watchdog 2.0] Activado. Timeout Inicial: {INITIAL_TIMEOUT_MINUTES} min. Timeout Operativo: {OPERATIONAL_TIMEOUT_MINUTES} min.")

    while True:
        time.sleep(10)
        current_limit_min = OPERATIONAL_TIMEOUT_MINUTES if has_rendered_any else INITIAL_TIMEOUT_MINUTES
        current_limit_sec = current_limit_min * 60

        # 1. Comprobación de Escudo Protector de ComfyUI (Cola y Ejecución)
        is_busy = False
        try:
            req_q = urllib.request.Request("http://127.0.0.1:8188/queue")
            with urllib.request.urlopen(req_q, timeout=4) as rq:
                q_data = json.loads(rq.read().decode("utf-8"))
                running = q_data.get("queue_running", [])
                pending = q_data.get("queue_pending", [])
                if len(running) > 0 or len(pending) > 0:
                    is_busy = True
                    last_activity = time.time()
                    warning_emitted = False
                    # Mantener actualizado en Supabase que está renderizando
                    register_supabase(url, status="running", note=f"Renderizando activo ({len(running)} ejecutando, {len(pending)} en espera)")
        except Exception:
            pass

        # 2. Comprobar Historial de Trabajos Completados
        try:
            req_h = urllib.request.Request("http://127.0.0.1:8188/history")
            with urllib.request.urlopen(req_h, timeout=4) as rh:
                history = json.loads(rh.read().decode("utf-8"))
                count = len(history)
                if count > last_job_count:
                    diff = count - last_job_count
                    last_job_count = count
                    total_videos_rendered += diff
                    last_activity = time.time()
                    has_rendered_any = True
                    warning_emitted = False
                    print(f"⚡ [Watchdog 2.0] Render completado con éxito (Total sesión: {total_videos_rendered}). Cambiando a régimen operativo de {OPERATIONAL_TIMEOUT_MINUTES} min.")
                    register_supabase(url, status="running", note=f"Régimen activo ({OPERATIONAL_TIMEOUT_MINUTES}m) | Total renders: {total_videos_rendered}")
        except Exception:
            pass

        if is_busy:
            continue

        # 3. Comprobar Pings de Presencia o Señal de Apagado Manual desde el Frontend
        try:
            req_db = urllib.request.Request(
                f"{SUPABASE_URL}/rest/v1/ai_jobs?id=eq.{TUNNEL_NODE_ID}&select=prompt",
                headers={"apikey": SUPABASE_KEY, "Authorization": f"Bearer {SUPABASE_KEY}"}
            )
            with urllib.request.urlopen(req_db, timeout=4) as rdb:
                db_data = json.loads(rdb.read().decode("utf-8"))
                if db_data and len(db_data) > 0:
                    prompt_txt = str(db_data[0].get("prompt", ""))
                    if "KEEPALIVE" in prompt_txt or "TOUCH" in prompt_txt:
                        last_activity = time.time()
                        warning_emitted = False
                        print("💓 [Watchdog 2.0] Presencia humana confirmada desde Frontend. Reloj reseteado.")
                        register_supabase(url, status="running", note=f"Presencia activa | Timeout: {current_limit_min}m")
                    elif "MANUAL_SHUTDOWN" in prompt_txt:
                        print("🛑 [Watchdog 2.0] Orden de apagado manual recibida desde Frontend.")
                        write_audit_session_log(url, started_session_at, total_videos_rendered, "USER_MANUAL_SHUTDOWN")
                        register_supabase(url, status="offline", note="Apagado manual confirmado por usuario")
                        shutdown_event.set()
                        break
        except Exception:
            pass

        idle_sec = time.time() - last_activity

        # 4. Fase de Preaviso / Warning (120 segundos antes del apagado)
        remaining_sec = max(0, int(current_limit_sec - idle_sec))
        if idle_sec >= (current_limit_sec - WARNING_WINDOW_SECONDS) and not warning_emitted:
            warning_emitted = True
            print(f"⚠️ [Watchdog 2.0] Preaviso de apagado inminente: {remaining_sec}s restantes. Notificando al frontend...")
            register_supabase(url, status="warning", note=f"PREAVISO_INACTIVIDAD:{remaining_sec}")

        # 5. Ejecución del Apagado por Inactividad
        if idle_sec >= current_limit_sec:
            print(f"🛑 [Watchdog 2.0] INACTIVIDAD CONFIRMADA: Han pasado {current_limit_min} minutos sin respuesta de usuario.")
            reason = "AUTO_IDLE_TIMEOUT_15M_NO_RESPONSE" if has_rendered_any else "AUTO_IDLE_TIMEOUT_35M_INITIAL_NO_RESPONSE"
            write_audit_session_log(url, started_session_at, total_videos_rendered, reason)
            register_supabase(url, status="offline", note=f"Apagado automático por inactividad ({reason})")
            shutdown_event.set()
            break

threading.Thread(target=watchdog_loop, daemon=True).start()

# Espera activa y limpia de la señal de apagado del Watchdog
try:
    shutdown_event.wait()
    print("🛑 [Watchdog 2.0] Señal de apagado recibida. Cerrando túnel y procesos limpiamente...")
    try:
        proc.terminate()
    except Exception:
        pass
    subprocess.run(["pkill", "-f", "cloudflared"])
    subprocess.run(["pkill", "-f", "main.py"])
    print("🎉 [Watchdog 2.0] Apagado limpio exitoso. Horas de GPU protegidas con cero desperdicio.")
except KeyboardInterrupt:
    print("🛑 Servidor detenido manualmente.")
    write_audit_session_log(url, started_session_at, 0, "TERMINATED_BY_KEYBOARD")
    register_supabase(url, status="offline", note="Detenido manualmente por usuario")

